# Uber Ride Demand — Forecasting & Clustering (v2)

**Extends:** `UberTripAnalysis.ipynb` (descriptive analytics) with **predictive & unsupervised modeling**.

**Business questions:**
1. Can we **forecast daily trip demand** reliably enough to plan driver supply? *(Forecasting — SARIMA, MAE/RMSE/MAPE)*
2. Are there natural **demand archetypes** across zone × hour cells? *(Clustering — K-Means, silhouette validation)*

**Data:** `Uber_Cleaned_data.csv` — 35,000 trips, Jan–Dec 2024, 5 pickup zones, 24 hourly buckets.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf

sns.set_theme(style="whitegrid")
DATA_PATH = "data/Uber_Cleaned_data.csv"
df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
print(df.shape)
df.head()

In [ ]:
# Basic integrity checks before modelling
assert df["Trip_ID"].is_unique, "duplicate trip ids"
assert df[["Date", "Hour", "Pickup_Location", "Fare_Amount"]].isna().sum().sum() == 0, "nulls in key columns"
print("rows:", len(df), "| date range:", df.Date.min().date(), "->", df.Date.max().date())
print("zones:", sorted(df.Pickup_Location.unique()))

## 1 · Clustering — Demand Archetypes (Zone × Hour cells)

Each **zone-hour cell** (5 zones × 24 hours = 120 cells) is profiled by demand volume, fare, distance and duration.
K-Means groups these cells into interpretable **demand archetypes**; *k* is selected with the silhouette score.

In [ ]:
cell = (df.groupby(["Pickup_Location", "Hour"])
          .agg(trips=("Trip_ID", "count"),
               avg_fare=("Fare_Amount", "mean"),
               avg_distance=("Trip_Distance_km", "mean"),
               avg_duration=("Trip_Duration_min", "mean"))
          .reset_index())
X = StandardScaler().fit_transform(cell[["trips", "avg_fare", "avg_distance", "avg_duration"]])

scores = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    scores[k] = silhouette_score(X, km.labels_)
best_k = max(scores, key=scores.get)
print("silhouette scores:", {k: round(v, 3) for k, v in scores.items()}, "| best k =", best_k)

In [ ]:
km = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit(X)
cell["cluster"] = km.labels_
summary = (cell.groupby("cluster")
             .agg(cells=("trips", "size"), total_trips=("trips", "sum"),
                  avg_fare=("avg_fare", "mean"), avg_distance=("avg_distance", "mean"))
             .round(2))
print(summary)
display(cell.groupby("cluster")["Hour"].apply(lambda h: sorted(h.unique())))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for c, grp in cell.groupby("cluster"):
    axes[0].plot(grp.Hour, grp.trips, marker="o", label=f"cluster {c}")
axes[0].set(title="Hourly demand profile by cluster", xlabel="hour", ylabel="avg trips per cell")
axes[0].legend()
sns.heatmap(cell.pivot(index="Hour", columns="Pickup_Location", values="cluster"),
            ax=axes[1], cmap="tab10", cbar_kws={"label": "cluster"})
axes[1].set_title("Zone × Hour cluster map")
fig.tight_layout(); fig.savefig("image/clustering_profiles.png", dpi=150)
plt.show()
cell.to_csv("outputs/demand_cluster_cells.csv", index=False)

## 2 · Forecasting — Daily Trip Demand (SARIMA)

Daily trips for 2024 with weekly seasonality (s=7). Train: Jan–Oct · Test: Nov–Dec.
Metrics: **MAE, RMSE, MAPE**. A naive 7-day seasonal-naive baseline is reported for honesty.

In [ ]:
daily = (df.groupby("Date").size().rename("trips").to_frame()
           .reindex(pd.date_range(df.Date.min(), df.Date.max()), fill_value=0)
           .rename_axis("Date"))
split = int(len(daily) * 0.833)  # ~10 months train
train, test = daily.iloc[:split], daily.iloc[split:]
print(f"train: {len(train)} days | test: {len(test)} days | mean daily trips: {daily.trips.mean():.0f}")

In [ ]:
model = SARIMAX(train.trips, order=(1, 1, 1), seasonal_order=(1, 0, 1, 7),
                enforce_stationarity=False, enforce_invertibility=False)
res = model.fit(disp=False)
pred = res.get_forecast(len(test)).predicted_mean.clip(lower=0)

mae = np.mean(np.abs(test.trips - pred))
rmse = float(np.sqrt(np.mean((test.trips - pred) ** 2)))
mape = float(np.mean(np.abs(test.trips - pred) / test.trips) * 100)
naive = np.mean(np.abs(test.trips - daily.trips.shift(7).iloc[split:].values))
print(f"SARIMA  -> MAE {mae:.1f} | RMSE {rmse:.1f} | MAPE {mape:.1f}%")
print(f"Seasonal-naive baseline -> MAE {naive:.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(train.index, train.trips, label="train", lw=0.8)
ax.plot(test.index, test.trips, label="actual", lw=0.9)
ax.plot(test.index, pred, "--", label="SARIMA forecast", color="crimson")
ax.set(title="Daily trip demand — held-out forecast", ylabel="trips/day")
ax.legend()
fig.tight_layout(); fig.savefig("image/forecast_daily_demand.png", dpi=150)
plt.show()

### Residual diagnostics — validating the forecast
The JD emphasises *disciplined validation*. Before trusting the MAPE, we check the residuals: they should be **mean-zero, free of autocorrelation (Ljung-Box), and roughly normal**.

In [ ]:
# Residual diagnostics — is the model capturing the structure, or is signal left on the table?
resid = (test.trips - pred).rename("residual")
print(f"residual mean: {resid.mean():.2f} (unbiased if ~0) | std: {resid.std():.2f}")
lb = acorr_ljungbox(resid, lags=[7, 14], return_df=True)
print("Ljung-Box p-values (want > 0.05 = no leftover autocorrelation):")
print(lb[["lb_stat", "lb_pvalue"]].round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].plot(test.index, resid); axes[0].axhline(0, color="crimson", ls="--")
axes[0].set(title="Residuals over time")
plot_acf(resid, lags=20, ax=axes[1])
axes[1].set(title="Residual ACF")
from scipy import stats as sps
sps.probplot(resid, plot=axes[2]); axes[2].set(title="Q-Q plot")
fig.tight_layout(); fig.savefig("image/forecast_residuals.png", dpi=150)
plt.show()

In [ ]:
# Refit on all data -> next 14-day operational forecast
full = SARIMAX(daily.trips, order=(1, 1, 1), seasonal_order=(1, 0, 1, 7),
               enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
fut = full.get_forecast(14)
fut_mean = fut.predicted_mean.clip(lower=0)
ci = fut.conf_int(alpha=0.05).clip(lower=0)
print("Next-14-day avg daily demand:", round(fut_mean.mean()))

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(daily.index[-60:], daily.trips[-60:], label="actual")
ax.plot(fut_mean.index, fut_mean, "--", color="crimson", label="14-day forecast")
ax.fill_between(ci.index, ci.iloc[:, 0], ci.iloc[:, 1], alpha=0.15, color="crimson", label="95% CI")
ax.set(title="Operational forecast — next 14 days", ylabel="trips/day"); ax.legend()
fig.tight_layout(); fig.savefig("image/forecast_14day.png", dpi=150)
plt.show()

## 3 · Findings & Limitations

- **Clustering:** zone × hour cells group into interpretable demand archetypes (peak-core vs off-peak vs airport-driven cells — see cluster map). Use them to align driver repositioning windows rather than one city-wide policy.
- **Forecasting:** SARIMA captures the weekly cycle; report MAE/RMSE/MAPE vs the seasonal-naive baseline honestly — if the lift is small, demand here is largely calendar-driven and staffing should follow the hourly profile from the existing EDA (5–7 PM peak).
- **Limitations:** single year of data → no annual seasonality testable; point predictions in low-volume periods carry wide 95% CI; no weather/event covariates included.

*Reproducible: fixed random_state, deterministic train/test split, all metrics printed from held-out data.*